# CP2/CP3 — Anonimizador LGPD (Qwen3-0.6B + LoRA) — **alternativa no Google Colab**

> O projeto foi treinado e avaliado **localmente, numa GTX 1060 6 GB** (ver README). Este notebook é uma
> **alternativa para quem não tem GPU NVIDIA**: reproduz o treino numa GPU T4 do Colab (fp16, mais rápido).

1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha **GPU (T4)**.
2. Rode as células em ordem. Os resultados são salvos no Google Drive; se a sessão cair, rode de novo:
   os experimentos já concluídos são pulados.
3. Na seção "1. Escolha o modo", escolha entre o **modo rápido** (só o modelo final, exp7; estimativa de ~30 min na T4) e o **completo**
   (as 3 rodadas: 7 treinos + avaliações; estimativa de ~2h na T4).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cp-llm-lgpd'
!mkdir -p "$DRIVE_DIR"

In [ ]:
%cd /content
import os
if not os.path.exists('/content/cp-llm-lgpd'):
    !git clone https://github.com/ValottoEngineer/cp-llm-lgpd.git
%cd /content/cp-llm-lgpd
!pip install -q -r requirements-colab.txt -r requirements-dev.txt
!python -m pytest -q tests

Restaura resultados de uma sessão anterior (se houver).

In [ ]:
!mkdir -p outputs results
!cp -rn "$DRIVE_DIR/outputs/." outputs/ 2>/dev/null || true
!cp -rn "$DRIVE_DIR/results/." results/ 2>/dev/null || true
!python scripts/gpu_check.py

## 1. Escolha o modo

Os datasets (v1, v2, v3 e as perguntas gerais da autodestilação) e os três desafios já vêm no repositório.

In [ ]:
MODO = 'rapido'   # 'rapido' = só o exp7 (modelo final) | 'completo' = as 3 rodadas (exp1 a exp7)

## 2. Treino e avaliação

In [ ]:
try:
    if MODO == 'rapido':
        !python scripts/experimentos.py --so exp7_r16_lr2e-4_ep2_v3
    else:
        !python scripts/experimentos.py
    !python scripts/pos_processamento.py
    !python scripts/resumo.py > /dev/null
finally:
    !mkdir -p "$DRIVE_DIR/outputs" "$DRIVE_DIR/results"
    !cp -r outputs/. "$DRIVE_DIR/outputs/"
    !cp -r results/. "$DRIVE_DIR/results/"

## 3. Resultados

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/resumo.md', encoding='utf-8').read()))
display(Markdown(open('results/regex_resumo.md', encoding='utf-8').read()))
if os.path.exists('results/curvas_loss.png'):
    display(Image('results/curvas_loss.png'))

## 4. Baixar o adaptador e os resultados

Descompacte na raiz do projeto local (pastas `outputs/melhor` e `results/`) e rode `python app/app.py`.

In [ ]:
!zip -qr resultados_colab.zip outputs/melhor results
!cp resultados_colab.zip "$DRIVE_DIR/"
from google.colab import files
files.download('resultados_colab.zip')

Ao terminar: **Ambiente de execução → Desconectar e excluir ambiente de execução**, para não gastar unidades de computação.